# Treinamento dos Modelos Baseline: MLP e VGG no CIFAR-10

Este notebook implementa o treinamento completo e a avaliação dos modelos baseline **CIFAR10MLP** (totalmente conectado) e **CIFAR10VGG** (convolucional profundo) utilizando o dataset CIFAR-10.

**Objetivos:**
1. Carregar a divisão determinística de treino/validação gerada anteriormente (`data/cifar10_split.pt`).
2. Instanciar e treinar os baselines MLP e VGG até a convergência utilizando o otimizador Adam.
3. Avaliar os baselines treinados no conjunto de teste oficial usando métricas de classificação detalhadas (acurácia, precisão, sensibilidade, F1-score).
4. Salvar os modelos na pasta `model_dir/` para servir como pontos de partida nos pipelines de compressão por poda (pruning) subsequentes.

## 1. Configuração e Importações

Carregamos as bibliotecas do PyTorch e importamos as classes de rede (`CIFAR10MLP`, `CIFAR10VGG`) e funções de treinamento e avaliação a partir do arquivo modular `utils.py`.

In [ ]:
import torch
import torch.nn as nn
import torchvision
from torchvision.transforms import v2
import matplotlib.pyplot as plt
import numpy as np
import random
from pathlib import Path
from torch.utils.data import DataLoader, Subset

# Importar modelos e rotinas de utils.py
from utils import (
    CIFAR10MLP,
    CIFAR10VGG,
    train_model,
    evaluate_model,
    compute_metrics,
    classes
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo ativo: {device}")

# Garantir reprodutibilidade
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## 2. Carregamento de Dados e Split Determinístico

Recuperamos a divisão determinística treino/validação que salvamos no disco anteriormente para garantir consistência em todos os treinamentos e podas futuras.

In [ ]:
transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2470, 0.2435, 0.2616)
    )
])

full_trainset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform
)
testset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform
)

# Recarregar divisão original
split = torch.load("data/cifar10_split.pt", weights_only=False)
train_indices = split["train_indices"]
val_indices = split["val_indices"]

trainset = Subset(full_trainset, train_indices)
valset = Subset(full_trainset, val_indices)

batch_size = 64
trainloader = DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)
valloader = DataLoader(valset, batch_size=batch_size, shuffle=False, num_workers=2)
testloader = DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)

print(f"Treino: {len(trainset):,} | Validação: {len(valset):,} | Teste: {len(testset):,}")

## 3. Treinamento da Rede Totalmente Conectada (CIFAR10MLP)

O modelo MLP possui camadas lineares densas que serão podadas posteriormente. Treinamos este modelo baseline por 15 épocas.

In [ ]:
mlp_model = CIFAR10MLP()
print(mlp_model)
print(f"Total de parâmetros ativos: {sum(p.numel() for p in mlp_model.parameters()):,}")

# Treinar o modelo baseline
print("\nIniciando treinamento do baseline MLP...")
mlp_model, mlp_train_losses, mlp_val_losses, mlp_val_accuracies = train_model(
    mlp_model,
    trainloader,
    valloader,
    epochs=15,
    lr=1e-3
)

### Avaliação e Salvamento do Baseline MLP

Avaliamos o modelo treinado no conjunto de teste e salvamos seus pesos em `model_dir/cifar_mlp.pt`.

In [ ]:
models_dir = Path('model_dir')
models_dir.mkdir(exist_ok=True)

torch.save(mlp_model, models_dir / 'cifar_mlp.pt')
print("Modelo MLP salvo com sucesso em model_dir/cifar_mlp.pt")

# Obter métricas de teste
y_true, y_pred, y_probs = evaluate_model(mlp_model, testloader, device)
mlp_metrics = compute_metrics(y_true, y_pred)

print("\nMétricas de Teste do MLP Baseline:")
for metric_name, score in mlp_metrics.items():
    print(f"  {metric_name:<20}: {score:.4f}")

## 4. Treinamento da Rede Convolucional Profunda (CIFAR10VGG)

A VGG-11 modificada é ideal para testes de poda estrutural de canais convolucionais (filtros) devido à sua alta complexidade de parâmetros. Treinamos o baseline convolucional.

In [ ]:
vgg_model = CIFAR10VGG()
print(vgg_model)
print(f"Total de parâmetros ativos: {sum(p.numel() for p in vgg_model.parameters()):,}")

# Treinar o modelo baseline
print("\nIniciando treinamento do baseline VGG...")
vgg_model, vgg_train_losses, vgg_val_losses, vgg_val_accuracies = train_model(
    vgg_model,
    trainloader,
    valloader,
    epochs=15,
    lr=1e-3
)

### Avaliação e Salvamento do Baseline VGG

Avaliamos o modelo convolucional treinado no conjunto de teste e salvamos em `model_dir/cifar_vgg.pt`.

In [ ]:
torch.save(vgg_model, models_dir / 'cifar_vgg.pt')
print("Modelo VGG salvo com sucesso em model_dir/cifar_vgg.pt")

# Obter métricas de teste
y_true, y_pred, y_probs = evaluate_model(vgg_model, testloader, device)
vgg_metrics = compute_metrics(y_true, y_pred)

print("\nMétricas de Teste do VGG Baseline:")
for metric_name, score in vgg_metrics.items():
    print(f"  {metric_name:<20}: {score:.4f}")

## 5. Comparação e Curvas de Aprendizado

Comparamos quantitativamente as métricas de ambos os modelos baseline e plotamos as curvas de perda (loss) de treinamento e validação para analisar o comportamento da convergência.

In [ ]:
import pandas as pd

# Criar DataFrame de comparação
comparison_df = pd.DataFrame({
    "MLP Baseline": mlp_metrics,
    "VGG Baseline": vgg_metrics
})

print("\nTabela Comparativa de Métricas no Teste:")
display(comparison_df)

# Plotar perdas
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(mlp_train_losses, label="Treino MLP", color="#1F77B4", linewidth=2)
plt.plot(mlp_val_losses, label="Validação MLP", color="#FF7F0E", linestyle="--", linewidth=2)
plt.title("Curvas de Loss - MLP Baseline", fontsize=13, fontweight='bold')
plt.xlabel("Época", fontsize=11)
plt.ylabel("Loss", fontsize=11)
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(vgg_train_losses, label="Treino VGG", color="#2CA02C", linewidth=2)
plt.plot(vgg_val_losses, label="Validação VGG", color="#D62728", linestyle="--", linewidth=2)
plt.title("Curvas de Loss - VGG Baseline", fontsize=13, fontweight='bold')
plt.xlabel("Época", fontsize=11)
plt.ylabel("Loss", fontsize=11)
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()